# Препроцессинг данных

Получаем плоскую таблицу `data/processed/matches.csv`, готовую для feature engineering

### Преобразования

1. Убираем матчи со статусами `bye` и `walkover`, убрать матчи без `winner`
2. **`score_json` → плоские столбцы:** геймы по сетам; в скобках у одной стороны — очки **проигравшего в тайбрейке**; очки победителя в TB: 7 если проигравший набрал < 6, иначе проигравший + 2; `set{i}_tb_team*` хранят полный счёт TB по сторонам; победитель сета по геймам
3. **`stats_json` → плоские столбцы:** 13 метрик × 2 команды для уровней `match`, `set_1`, `set_2`, `set_3`. Проценты переводим в float, целые в int
4. **`pbp_json`:** пока не трогаем, убираем
5. **Прочие столбцы:** `duration` переводим в минуты, `00:00` заменяем на NaN; добавить `tournament_level` из `tournaments.csv`. удалить неинформативные столбцы (`category`, `started_time`, `schedule_label`, `court`, `court_order`, `index`, `name`, `pbp_json`)
6. **Сохраняем в:** `data/processed/matches.csv`

In [61]:
import pandas as pd
import json
import re
from pathlib import Path

RAW = Path("../../data/raw")
PROCESSED = Path("../../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

matches = pd.read_csv(RAW / "men/matches.csv", parse_dates=["played_at"])
tournaments = pd.read_csv(RAW / "tournaments.csv")

print(f"Исходных матчей: {len(matches)}")

Исходных матчей: 2616


## 1. Фильтрация

In [62]:
before = len(matches)
matches = matches[matches["status"].isin(["finished"])].copy()
matches = matches[matches["winner"].notna()].copy()
after = len(matches)
print(f"Отфильтровано: {before} → {after} (убрано {before - after} матчей: walkovers, byes, retired, без winner)")

Отфильтровано: 2616 → 2549 (убрано 67 матчей: walkovers, byes, retired, без winner)


## 2. Раскрытие `score_json`

In [63]:
def parse_side_score(val):
    """'6(1)' → геймы 6, в скобках очки этой стороны в TB (проигравший в тайбрейке). '7' → 7, без TB"""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None, None
    s = str(val).strip()
    m = re.match(r"^(\d+)(?:\((\d+)\))?$", s)
    if not m:
        return None, None
    games = int(m.group(1))
    tb_loser_pts = int(m.group(2)) if m.group(2) is not None else None
    return games, tb_loser_pts


def tb_winner_points(loser_pts):
    """Победитель TB: до 7 с отрывом; если проигравший >= 6 очков, у победителя на 2 больше"""
    if loser_pts is None:
        return None
    if loser_pts < 6:
        return 7
    return loser_pts + 2


def expand_score(score_json):
    result = {}
    if pd.isna(score_json):
        return result

    sets = json.loads(score_json)
    result["n_sets"] = len(sets)

    for i, s in enumerate(sets, start=1):
        g1, p1 = parse_side_score(s.get("team_1"))
        g2, p2 = parse_side_score(s.get("team_2"))
        result[f"set{i}_team1"] = g1
        result[f"set{i}_team2"] = g2

        tb1, tb2 = None, None
        if p1 is not None and p2 is None:
            tb1 = p1
            tb2 = tb_winner_points(p1)
        elif p2 is not None and p1 is None:
            tb2 = p2
            tb1 = tb_winner_points(p2)
        elif p1 is not None and p2 is not None:
            tb1, tb2 = p1, p2

        result[f"set{i}_tb_team1"] = tb1
        result[f"set{i}_tb_team2"] = tb2
        if g1 is not None and g2 is not None:
            result[f"set{i}_winner"] = "team_1" if g1 > g2 else "team_2" if g2 > g1 else None

    return result


score_df = matches["score_json"].apply(lambda x: pd.Series(expand_score(x)))
matches = pd.concat([matches, score_df], axis=1)

print("Новые столбцы из score_json:")
print([c for c in score_df.columns])
print()
print("n_sets distribution:")
print(matches["n_sets"].value_counts().sort_index())

Новые столбцы из score_json:
['n_sets', 'set1_team1', 'set1_team2', 'set1_tb_team1', 'set1_tb_team2', 'set1_winner', 'set2_team1', 'set2_team2', 'set2_tb_team1', 'set2_tb_team2', 'set2_winner', 'set3_team1', 'set3_team2', 'set3_tb_team1', 'set3_tb_team2', 'set3_winner']

n_sets distribution:
n_sets
2    1839
3     710
Name: count, dtype: int64


## 3. Раскрытие `stats_json`

In [64]:
STAT_KEYS = [
    "total_points_won", "break_points_converted", "longest_streak",
    "aces", "double_faults", "won_on_1st_serve", "won_on_2nd_serve",
    "service_games", "won_on_1st_return", "won_on_2nd_return",
    "return_games", "total_won_on_serve", "total_won_on_return",
]

STAT_LEVELS = ["match", "set_1", "set_2", "set_3"]


def parse_stat_value(val):
    """'54%' → 0.54, '5' → 5.0, '' → NaN"""
    if not val or val == "":
        return None
    if val.endswith("%"):
        return float(val[:-1]) / 100.0
    return float(val)


def expand_stats(stats_json):
    result = {}
    if pd.isna(stats_json):
        return result

    data = json.loads(stats_json)

    for level in STAT_LEVELS:
        level_data = data.get(level)
        if not level_data or not isinstance(level_data, dict):
            continue
        for stat in STAT_KEYS:
            pair = level_data.get(stat, {})
            if not isinstance(pair, dict):
                continue
            for team in ["team_1", "team_2"]:
                col = f"stats_{level}_{stat}_{team}"
                result[col] = parse_stat_value(pair.get(team, ""))

    return result


stats_df = matches["stats_json"].apply(lambda x: pd.Series(expand_stats(x)))
matches = pd.concat([matches, stats_df], axis=1)

print(f"Столбцов из stats_json: {len(stats_df.columns)}")
print(f"Пример: {list(stats_df.columns[:6])}")

Столбцов из stats_json: 104
Пример: ['stats_match_total_points_won_team_1', 'stats_match_total_points_won_team_2', 'stats_match_break_points_converted_team_1', 'stats_match_break_points_converted_team_2', 'stats_match_longest_streak_team_1', 'stats_match_longest_streak_team_2']


## 4. Обработка прочих столбцов

In [65]:
def duration_to_minutes(val):
    if pd.isna(val) or val == "00:00":
        return None
    parts = val.split(":")
    return int(parts[0]) * 60 + int(parts[1])


matches["duration_min"] = matches["duration"].apply(duration_to_minutes)

matches["year"] = matches["played_at"].dt.year
matches["month"] = matches["played_at"].dt.month

matches = matches.merge(
    tournaments[["tournament_id", "level"]].rename(columns={"level": "tournament_level"}),
    on="tournament_id",
    how="left",
)

print(f"duration_min заполнено: {matches['duration_min'].notna().sum()} из {len(matches)}")
print(f"tournament_level value_counts:")
print(matches["tournament_level"].value_counts())

duration_min заполнено: 848 из 2549
tournament_level value_counts:
tournament_level
p1              1127
major            559
p2               557
fip_platinum     292
finals            14
Name: count, dtype: int64


## 5. Удаление ненужных столбцов и сохранение

In [66]:
drop_cols = [
    "category", "name", "index", "schedule_label", "court", "court_order",
    "started_time", "duration", "score_json", "stats_json", "pbp_json",
]
matches = matches.drop(columns=drop_cols)

print(f"Итоговая таблица: {matches.shape[0]} строк, {matches.shape[1]} столбцов")
print()
print("Столбцы:")
for c in matches.columns:
    print(f"  {c}: {matches[c].dtype}")

Итоговая таблица: 2549 строк, 135 столбцов

Столбцы:
  match_id: int64
  tournament_id: int64
  player_id_1: int64
  player_id_2: int64
  player_id_3: int64
  player_id_4: int64
  round: int64
  round_name: object
  played_at: datetime64[ns]
  status: object
  winner: object
  n_sets: int64
  set1_team1: int64
  set1_team2: int64
  set1_tb_team1: float64
  set1_tb_team2: float64
  set1_winner: object
  set2_team1: int64
  set2_team2: int64
  set2_tb_team1: float64
  set2_tb_team2: float64
  set2_winner: object
  set3_team1: float64
  set3_team2: float64
  set3_tb_team1: float64
  set3_tb_team2: float64
  set3_winner: object
  stats_match_total_points_won_team_1: float64
  stats_match_total_points_won_team_2: float64
  stats_match_break_points_converted_team_1: float64
  stats_match_break_points_converted_team_2: float64
  stats_match_longest_streak_team_1: float64
  stats_match_longest_streak_team_2: float64
  stats_match_aces_team_1: float64
  stats_match_aces_team_2: float64
  stats_

In [67]:
matches.head()

,match_id,tournament_id,player_id_1,player_id_2,player_id_3,player_id_4,round,round_name,played_at,status,...,stats_set_3_return_games_team_1,stats_set_3_return_games_team_2,stats_set_3_total_won_on_serve_team_1,stats_set_3_total_won_on_serve_team_2,stats_set_3_total_won_on_return_team_1,stats_set_3_total_won_on_return_team_2,duration_min,year,month,tournament_level
0,1306,7,235,236,107,267,32,Round of 64,2023-02-28,finished,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,2,major
1,1307,7,233,246,98,237,32,Round of 64,2023-02-28,finished,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,2,major
2,1308,7,47,247,89,102,32,Round of 64,2023-02-28,finished,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,2,major
3,1309,7,68,76,248,249,32,Round of 64,2023-02-28,finished,...,4.0,5.0,0.68,0.59,0.41,0.32,NaN,2023,2,major
4,1310,7,82,105,307,308,32,Round of 64,2023-02-28,finished,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,2,major


In [68]:
matches.shape

(2549, 135)

In [71]:
matches.loc[2544].to_dict()

{'match_id': 7474,
 'tournament_id': 728,
 'player_id_1': 87,
 'player_id_2': 89,
 'player_id_3': 78,
 'player_id_4': 103,
 'round': 4,
 'round_name': 'Quarter',
 'played_at': Timestamp('2026-03-27 00:00:00'),
 'status': 'finished',
 'winner': 'team_1',
 'n_sets': 2,
 'set1_team1': 7,
 'set1_team2': 6,
 'set1_tb_team1': 8.0,
 'set1_tb_team2': 6.0,
 'set1_winner': 'team_1',
 'set2_team1': 6,
 'set2_team2': 4,
 'set2_tb_team1': nan,
 'set2_tb_team2': nan,
 'set2_winner': 'team_1',
 'set3_team1': nan,
 'set3_team2': nan,
 'set3_tb_team1': nan,
 'set3_tb_team2': nan,
 'set3_winner': nan,
 'stats_match_total_points_won_team_1': 0.52,
 'stats_match_total_points_won_team_2': 0.48,
 'stats_match_break_points_converted_team_1': 0.6,
 'stats_match_break_points_converted_team_2': 0.33,
 'stats_match_longest_streak_team_1': 5.0,
 'stats_match_longest_streak_team_2': 6.0,
 'stats_match_aces_team_1': 0.0,
 'stats_match_aces_team_2': 0.0,
 'stats_match_double_faults_team_1': 0.0,
 'stats_match_double

In [74]:
out_path = PROCESSED / "matches.csv"
matches.to_csv(out_path, index=False)